# Predictor-Based Feedback for Nonlinear Systems with Input Delay
### Reproduction of Bekiaris-Liberis & Krstic (Automatica, 2016) — Example 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/bekiaris-liberis-krstic-2016/blob/main/python/notebook.ipynb)

---

## 1. Introduction

This notebook reproduces **Example 1** from:

> N. Bekiaris-Liberis and M. Krstic, "Stability of predictor-based feedback for nonlinear systems with distributed input delay," *Automatica*, vol. 70, pp. 195–203, 2016.

**Problem**: Stabilize a nonlinear system where the control input reaches the plant only after a delay $D > 0$.

**Key idea**: Use a *predictor* to estimate the plant state $D$ seconds into the future, then apply the delay-free control law to the predicted state.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import sys, os

# Add src to path
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'python', 'src'))
sys.path.insert(0, 'src')

from predictor import (
    heun_predictor, run_pde_simulation, compute_Gamma, run_uncompensated
)

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (12, 7),
    'font.size': 12,
    'lines.linewidth': 2,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'text.usetex': False,
})

print('Dependencies loaded.')

## 2. System Model

The plant dynamics (Eq. 28-29):

$$\dot{X}_1(t) = 2X_2(t) + U(t)$$
$$\dot{X}_2(t) = \frac{X_2(t) + U(t-D)}{U(t-D)^2 + 1}$$

The control input $U(t)$ arrives at the plant only after delay $D = 1$ second.

**Without delay ($D=0$)**, the control law $U = -2X_2 - X_1$ renders the $Z$-system:
$$\dot{Z}_1 = 2Z_2 + U, \quad \dot{Z}_2 = \frac{Z_2 + U}{U^2 + 1}$$
globally asymptotically stable.

**With delay ($D>0$)**, we need to "predict" the future state $Z(t)$ using the predictor formulas (Eq. 33-34).

In [ ]:
# Parameters
D = 1.0       # Delay [s]
N = 100       # Spatial grid points
t_end = 20.0  # Simulation time [s]
X0 = np.array([1.0, 1.0])  # Initial state

print(f'Parameters: D={D}, N={N}, t_end={t_end}, X0={X0}')

## 3. Controller Design: Predictor-Based Feedback

The predictor is computed by solving a **spatial ODE** (Type II predictor, Eq. 79-80):

$$\frac{\partial p_2}{\partial x} = \frac{p_2 + u(x,t)}{u(x,t)^2 + 1}, \quad p_2(0) = X_2(t)$$
$$\frac{\partial p_1}{\partial x} = 2p_2, \quad\quad\quad p_1(0) = X_1(t)$$

where $u(x,t) = U(t+x-D)$ is the actuator state, and $Z_j(t) = p_j(D,t)$.

We solve this with the **Heun method** (improved Euler, $O(\Delta x^2)$).

The control law is then:
$$U(t) = -2Z_2(t) - Z_1(t)$$

## 4. Simulation

In [ ]:
# Run PDE simulation
print('Running PDE simulation...')
t, X, U_hist, Z_hist = run_pde_simulation(D, N, t_end, X0)
Gamma = compute_Gamma(t, X, U_hist, D)

print(f'Simulation complete: {len(t)} time steps')
print(f'|X(t_end)| = {np.linalg.norm(X[-1]):.4e}')
print(f'max|U|     = {np.max(np.abs(U_hist)):.4f}')
print(f'Gamma decay: {(1 - Gamma[-1]/Gamma[0])*100:.4f}%')

In [ ]:
# Plot main results
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# Plant state
axes[0, 0].plot(t, X[:, 0], 'b-', label=r'$X_1$')
axes[0, 0].plot(t, X[:, 1], 'r--', label=r'$X_2$')
axes[0, 0].set(xlabel='t [s]', ylabel='X', title=f'Plant State (D={D}, N={N})')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, t_end)

# Control input
axes[0, 1].plot(t, U_hist, 'k-')
axes[0, 1].set(xlabel='t [s]', ylabel='U', title='Control Input')
axes[0, 1].set_xlim(0, t_end)

# Gamma indicator
axes[1, 0].semilogy(t, Gamma, 'b-')
axes[1, 0].set(xlabel='t [s]', ylabel=r'$\Gamma(t)$', title='Stability Indicator')
axes[1, 0].set_xlim(0, t_end)

# Predictor states
axes[1, 1].plot(t, Z_hist[:, 0], 'b-', label=r'$Z_1$')
axes[1, 1].plot(t, Z_hist[:, 1], 'r--', label=r'$Z_2$')
axes[1, 1].set(xlabel='t [s]', ylabel='Z', title='Predictor State')
axes[1, 1].legend()
axes[1, 1].set_xlim(0, t_end)

plt.tight_layout()
plt.show()

## 5. Why Delay Compensation Matters

What happens if we apply the delay-free control law $U(t) = -2X_2(t) - X_1(t)$ directly, without the predictor? The controller doesn't account for the 1-second delay, so its decisions are based on outdated state information.

In [ ]:
# Run uncompensated baseline
print('Running uncompensated baseline...')
t_uc, X_uc, U_uc = run_uncompensated(D, N, t_end, X0)
Gamma_uc = compute_Gamma(t_uc, X_uc, U_uc, D)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# State comparison
ax1.plot(t, X[:, 0], 'b-', label=r'$X_1$ (compensated)', linewidth=2)
ax1.plot(t, X[:, 1], 'b--', label=r'$X_2$ (compensated)', linewidth=2)
ax1.plot(t_uc, X_uc[:, 0], 'r-', label=r'$X_1$ (no predictor)', linewidth=2)
ax1.plot(t_uc, X_uc[:, 1], 'r--', label=r'$X_2$ (no predictor)', linewidth=2)
ax1.set(xlabel='t [s]', ylabel='State', title='Compensated vs Uncompensated')
ax1.legend(fontsize=9)
ax1.set_xlim(0, min(t_end, 10))

# Gamma comparison
ax2.semilogy(t, Gamma, 'b-', label='Compensated', linewidth=2)
ax2.semilogy(t_uc, Gamma_uc, 'r-', label='Uncompensated', linewidth=2)
ax2.set(xlabel='t [s]', ylabel=r'$\Gamma(t)$', title='Stability Indicator')
ax2.legend()
ax2.set_xlim(0, min(t_end, 10))

plt.tight_layout()
plt.show()

print(f'\nCompensated:   |X(t_end)| = {np.linalg.norm(X[-1]):.4e}')
print(f'Uncompensated: |X(t_end)| = {np.linalg.norm(X_uc[-1]):.4e}')

## 6. Interactive Demo

Adjust the delay $D$ and initial conditions to explore the system behavior.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    @widgets.interact(
        D_val=widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Delay D:'),
        x1_0=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.5, description='X1(0):'),
        x2_0=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.5, description='X2(0):'),
    )
    def interactive_sim(D_val=1.0, x1_0=1.0, x2_0=1.0):
        X0_int = np.array([x1_0, x2_0])
        if np.linalg.norm(X0_int) < 0.01:
            print('Initial state too close to origin. Adjust sliders.')
            return

        t_i, X_i, U_i, Z_i = run_pde_simulation(D_val, 100, 20.0, X0_int)
        Gamma_i = compute_Gamma(t_i, X_i, U_i, D_val)

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        axes[0].plot(t_i, X_i[:, 0], 'b-', label=r'$X_1$')
        axes[0].plot(t_i, X_i[:, 1], 'r--', label=r'$X_2$')
        axes[0].set(xlabel='t [s]', ylabel='State', title=f'D={D_val:.1f}, X0=[{x1_0},{x2_0}]')
        axes[0].legend()
        axes[0].set_xlim(0, 20)

        axes[1].semilogy(t_i, Gamma_i, 'b-')
        axes[1].set(xlabel='t [s]', ylabel=r'$\Gamma$', title='Stability Indicator')
        axes[1].set_xlim(0, 20)

        axes[2].plot(X_i[:, 0], X_i[:, 1], 'b-', linewidth=1)
        axes[2].plot(X_i[0, 0], X_i[0, 1], 'go', markersize=8)
        axes[2].plot(0, 0, 'k+', markersize=12, markeredgewidth=2)
        axes[2].set(xlabel=r'$X_1$', ylabel=r'$X_2$', title='Phase Portrait')
        axes[2].set_aspect('equal')

        plt.tight_layout()
        plt.show()

        print(f'|X(t_end)| = {np.linalg.norm(X_i[-1]):.4e}')

except ImportError:
    print('ipywidgets not available. Install with: pip install ipywidgets')
    print('Skipping interactive demo.')

## 7. Delay Parameter Sweep

In [ ]:
D_values = [0.5, 1.0, 1.5, 2.0]
colors = ['b', 'r', 'g', 'purple']
styles = ['-', '--', '-.', ':']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for D_val, c, s in zip(D_values, colors, styles):
    print(f'Running D = {D_val}...')
    t_s, X_s, U_s, _ = run_pde_simulation(D_val, 100, 20.0, X0)
    Gamma_s = compute_Gamma(t_s, X_s, U_s, D_val)
    X_norm = np.sqrt(X_s[:, 0]**2 + X_s[:, 1]**2)

    ax1.plot(t_s, X_norm, color=c, linestyle=s, label=f'D = {D_val}')
    ax2.semilogy(t_s, Gamma_s, color=c, linestyle=s, label=f'D = {D_val}')

ax1.set(xlabel='t [s]', ylabel='|X(t)|', title='State Norm vs Delay')
ax1.legend()
ax1.set_xlim(0, 20)

ax2.set(xlabel='t [s]', ylabel=r'$\Gamma(t)$', title='Stability Indicator vs Delay')
ax2.legend()
ax2.set_xlim(0, 20)

plt.tight_layout()
plt.show()

## 8. Validation: Analytical Check at t=0

With $U(\theta) = 0$ for $\theta \in [-D, 0]$, the predictor has analytical solutions:
$$Z_2(0) = e^D \cdot X_2(0), \quad Z_1(0) = X_1(0) + 2(e^D - 1) \cdot X_2(0)$$

In [ ]:
# Analytical vs numerical at t=0
D_test = 1.0
X1_test, X2_test = 1.0, 1.0

Z2_exact = np.exp(D_test) * X2_test
Z1_exact = X1_test + 2 * (np.exp(D_test) - 1) * X2_test

print('Convergence study: Heun predictor vs analytical')
print(f'{"N":>6s}  {"Z1 error":>12s}  {"Z2 error":>12s}')
print('-' * 34)

for N_test in [50, 100, 200, 500, 1000]:
    u_test = np.zeros(N_test)
    Z1_num, Z2_num = heun_predictor(X1_test, X2_test, u_test, D_test, N_test)
    print(f'{N_test:6d}  {abs(Z1_num - Z1_exact):12.4e}  {abs(Z2_num - Z2_exact):12.4e}')

print(f'\nExact: Z1 = {Z1_exact:.6f}, Z2 = {Z2_exact:.6f}')

---

## References

N. Bekiaris-Liberis and M. Krstic, "Stability of predictor-based feedback for nonlinear systems with distributed input delay," *Automatica*, vol. 70, pp. 195–203, 2016. DOI: [10.1016/j.automatica.2016.04.010](https://doi.org/10.1016/j.automatica.2016.04.010)